# Lab Tasks - Solutions

## Task 1 - JSON Parsing

Use Python to download a file containing metadata for member states of the EU in JSON format from the URL: 

http://mlg.ucd.ie/modules/python/eu.json

In [ ]:
import urllib.request
import urllib.error

url = "http://mlg.ucd.ie/modules/python/eu.json"

try:
    # fetch the specified URL
    response = urllib.request.urlopen(url)
    raw_json = response.read().decode("utf-8")
    print("Successfully downloaded EU data")
    print(raw_json)
except urllib.error.HTTPError as e:
    print(f"HTTP Error {e.code}: {e.reason}")
except urllib.error.URLError as e:
    print(f"Network Error: {e.reason}")
except Exception as e:
    print(f"Unexpected error: {e}")

Parse the JSON data that you have downloaded.

In [ ]:
import json

try:
    # try to parse the data
    eu_data = json.loads(raw_json)
    print("Successfully parsed EU JSON data")
    print(eu_data)
except json.JSONDecodeError as e:
    print(f"JSON Error: {e}")
    eu_data = None
except Exception as e:
    print(f"Error parsing JSON: {e}")

From the parsed JSON data, print a list of the all of the country names, along with the corresponding capital city.

In [ ]:
countries = eu_data["countries"]
print(f"Found {len(countries)} countries:")
for country in countries:
    # make sure the JSON has the format we expect
    if "name" in country and "capital_city" in country:
        print(f"{country["name"]} = {country["capital_city"]}")
    else:
        print("Warning: Country missing name or capital_city field")

Extract the population information for each member state.

In [ ]:
print("Population information:")
for country in countries:
    # make sure the JSON has the format we expect
    if "name" in country and "population" in country:
        print(f"{country["name"]} = {country["population"]}")
    else:
        print("Warning: Country missing population field")

Export the EU member state data to a new file in CSV format, with an appropriate header line.

In [ ]:
import csv

try:
    with open("eu.csv", "w", encoding="utf-8", newline="") as fout:
        # specify the ordered list of fields
        fields = ["name", "population", "capital_city", "first_language"]
        writer = csv.DictWriter(fout, fieldnames=fields)
        # write the header row
        writer.writeheader()
        # write each country record
        for country in countries:
            writer.writerow(country)
    print(f"Successfully exported {len(countries)} countries to eu.csv")
except IOError as e:
    print(f"File Error: Could not write to eu.csv - {e}")
except csv.Error as e:
    print(f"CSV Error: {e}")
except Exception as e:
    print(f"Unexpected error: {e}")

## Task 2 - XML Parsing

Use Python to download a file containing a contact list in XML format from the URL: 

http://mlg.ucd.ie/modules/python/contacts.xml

In [ ]:
url = "http://mlg.ucd.ie/modules/python/contacts.xml"

try:
    response = urllib.request.urlopen(url)
    raw_xml = response.read().decode("utf-8")
    print("Successfully downloaded XML data")
    print(raw_xml)
except urllib.error.HTTPError as e:
    print(f"HTTP Error {e.code}: {e.reason}")
except urllib.error.URLError as e:
    print(f"Network Error: {e.reason}")
except Exception as e:
    print(f"Unexpected error: {e}")

Parse the XML data that you have downloaded.

In [ ]:
import xml.etree.ElementTree

try:
    # parse the XML string to build a tree data structure
    tree = xml.etree.ElementTree.fromstring(raw_xml)
    print("Successfully parsed XML data")
except xml.etree.ElementTree.ParseError as e:
    print(f"XML Parse Error: {e}")
    tree = None
except Exception as e:
    print(f"Unexpected error parsing XML: {e}")
    tree = None

From the parsed XML data, extract the name, email address and phone number of each contact. 

Store these contacts in a list of dictionaries, and print them out.

In [ ]:
contacts = []
for entry in tree.findall("contact"):
    contact = {}
    
    # extract each field with error checking
    name_elem = entry.find("name")
    email_elem = entry.find("email")
    phone_elem = entry.find("phone")
    
    # check if all required elements exist
    if all(elem is not None and elem.text for elem in [name_elem, email_elem, phone_elem]):
        contact["name"] = name_elem.text.strip()
        contact["email"] = email_elem.text.strip()
        contact["phone"] = phone_elem.text.strip()
        contacts.append(contact)
    else:
        print("Warning: Skipping contact with missing or empty fields")

print(f"Successfully parsed {len(contacts)} complete contacts")

# display contacts
print("Extracted contacts:")
for i, contact in enumerate(contacts, 1):
    print(f"{i}. {contact['name']}:\temail={contact['email']}\tphone={contact['phone']}")

Export the contact data to a new file in CSV format.

In [ ]:
try:
    with open("contacts.csv", "w", encoding="utf-8", newline="") as fout:
        # specify the ordered list of fields
        fields = ["name", "email", "phone"]
        writer = csv.DictWriter(fout, fieldnames=fields)
        # write the header row
        writer.writeheader()
        # write each contact record
        for contact in contacts:
            writer.writerow(contact)
    print(f"Successfully exported {len(contacts)} contacts to contacts.csv")
    
except IOError as e:
    print(f"File Error: Could not write file - {e}")
except csv.Error as e:
    print(f"CSV Error: {e}")
except Exception as e:
    print(f"Unexpected error: {e}")

Use Pickle to serialise the contact data to a file. Verify that you can deseralise the data again.

In [ ]:
import pickle

try:
    # use Pickle to write the contact data out to a file
    with open("contacts.pkl", "wb") as fout:
        pickle.dump(contacts, fout)
    print(f"Successfully pickled {len(contacts)} contacts to contacts.pkl")
    
except IOError as e:
    print(f"File Error: Could not write file - {e}")
except Exception as e:
    print(f"Unexpected error: {e}")

In [ ]:
# verify that pickling worked by reading it back in again
try:
    with open("contacts.pkl", "rb") as fin:
        backup = pickle.load(fin)
    print(f"Successfully loaded {len(backup)} contacts from pickle file")
    
    print("Loaded contacts:")
    # process all the contacts
    for i, contact in enumerate(backup, 1):
        # check we have all the required fields for each contact
        if all(key in contact for key in ["name", "email", "phone"]):
            print(f"{i}. {contact['name']}:\temail={contact['email']}\tphone={contact['phone']}")
        else:
            print(f"{i}. Warning: Contact missing required fields")
            
except IOError as e:
    print(f"File Error: {e}")
except pickle.UnpicklingError as e:
    print(f"Pickle Error: {e}")
except Exception as e:
    print(f"Unexpected error: {e}")